# NOTEBOOK_Data-eau_00_API

### **Descriptif du Notebook VERSION VS CODE** : 

**Le projet :**
- Connexion à l'API Hub Eau sur le jeu de donnée "Piézométrie" (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Données recherchées :  
    A. La liste des stations autour de Valence 50km `/stations`  
    B. Les station et leur niveau d'eau. `/chroniques`
- Zone géographique : le bassion de Valence (26000) = utiliser une bbox « carré de 100 km centré sur Valence »

**Les étapes du Notebook :**

1. Imports 
2. Configuration : avec variables du projet (.env) + variables métier :
    - fichier .env = URL de l'API , Token (Identifiant pour l'API, ici pas nécessaire)
    - constantes/variables métier = Latitude/Longtude de Valence, demi_coté_carré de la Bbox
3. Calcul de la bbox
4. Fonction d'appel à l'API : gestion des erreurs + pagination
5. Récupération des stations piézométriques
6. Mise en DataFrame et premier contrôle (shape, colonnes, valeurs manquantes)
7. Sauvegarde des données brutes (data/raw/)
    

### Prérequis avant d'utiliser le notebook

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Dans le terminal, venv activé (`(.venv)` visible) : `source .venv/bin/activate`
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env` et `.venv/` ne doivent jamais apparaître


In [1]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/.venv/bin/python


### **Appel API - phase A /stations:**

In [2]:
# 1. Imports des bibliothèques

# Bibliothèque standard (livrée avec Python)
import json                         # sauvegarde des données brutes
import math                         # calcul de la bbox (cosinus de la latitude)
import os                           # lecture des variables d'environnement
import time                         # pause entre deux appels à l'API
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
import pandas as pd                 # mise en DataFrame et contrôle des données
import requests                     # appels HTTP à l'API Hub'Eau
from dotenv import load_dotenv      # chargement du fichier .env


In [3]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # charge le .env (cherché en remontant les dossiers)

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_STATIONS = f"{BASE_URL}/stations"           # liste des stations piézométriques
URL_CHRONIQUES = f"{BASE_URL}/chroniques"       # mesures de niveau d'eau par station

# --- B. Constantes métier ---
VALENCE_LAT = 44.9334                           # latitude du centre de la zone (WGS84)
VALENCE_LON = 4.8924                            # longitude du centre de la zone (WGS84)
DEMI_COTE_KM = 50                               # demi-côté du carré => carré de 100 km

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE = 1000                                # nombre de résultats par page
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux pages (respect de l'API)

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)     # créé s'il n'existe pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Récapitulatif ---
print("URL stations   :", URL_STATIONS)
print("Centre         :", VALENCE_LAT, VALENCE_LON, f"(demi-côté {DEMI_COTE_KM} km)")
print("Racine projet  :", PROJECT_ROOT)
print("Données brutes :", DATA_RAW)


URL stations   : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations
Centre         : 44.9334 4.8924 (demi-côté 50 km)
Racine projet  : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau
Données brutes : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/raw


In [4]:
# 3. Calcul de la bbox

# Définir la zone concernée par les données = BBox autour de la ville choisie (Ici = Valence /50 km)
    # **Définir la zone BBox** (zone carré de 50 km autour de Valence avec code communes) : Rectangle d'emprise de l'objet demandé, emprise au format : min longitude, min latitude, max longitude, max latitude avec les coordonnées en WGS84 (EPSG:4326), le point doit être utilisé comme séparateur décimal, exemple : 1.6194,47.7965,2.1910,47.9988.  
    # Vérifier sur : https://bboxfinder.com/#44.484200,4.258000,45.382600,5.526800

def bbox_autour(lat, lon, demi_cote_km):
    """Renvoie la BBox au format Hub'Eau : min_lon,min_lat,max_lon,max_lat"""
    d_lat = demi_cote_km / 111.32
    d_lon = demi_cote_km / (111.32 * math.cos(math.radians(lat)))
    return f"{lon - d_lon:.4f},{lat - d_lat:.4f},{lon + d_lon:.4f},{lat + d_lat:.4f}"

BBOX = bbox_autour(VALENCE_LAT, VALENCE_LON, DEMI_COTE_KM)
print("BBox :", BBOX)

BBox : 4.2579,44.4842,5.5269,45.3826


In [5]:
# 4. Fonction d'appel à l'API : gestion des erreurs + pagination

def appeler_api(url, params=None, max_pages=50):
    """
    Interroge un endpoint Hub'Eau et récupère toutes les pages de résultats.

    Paramètres
    ----------
    url : str        URL de l'endpoint (ex. URL_STATIONS)
    params : dict    Paramètres de la requête (bbox, size, format...)
    max_pages : int  Sécurité : nombre maximum de pages à récupérer

    Renvoie
    -------
    list[dict]       L'ensemble des résultats (contenu des champs "data")
    """
    resultats = []
    num_page = 1

    while url:
        reponse = requests.get(url, params=params, timeout=TIMEOUT_S)

        # 200 = tout est là / 206 = résultat partiel, d'autres pages suivent
        if reponse.status_code not in (200, 206):
            raise RuntimeError(f"Erreur API {reponse.status_code} : {reponse.text[:300]}")

        contenu = reponse.json()
        resultats.extend(contenu["data"])
        print(f"Page {num_page} : {len(contenu['data'])} résultats "
              f"({len(resultats)}/{contenu['count']})")

        url = contenu.get("next")    # None sur la dernière page => fin de la boucle
        params = None                # l'URL "next" contient déjà tous les paramètres
        num_page += 1

        if num_page > max_pages:
            print(f"⚠️ Arrêt après {max_pages} pages : résultats incomplets")
            break
        if url:
            time.sleep(PAUSE_S)      # pause entre deux appels

    return resultats


In [6]:
# ---- TEST temporaire : vérifier la pagination avec de toutes petites pages ----
# test = appeler_api(URL_STATIONS, {"bbox": BBOX, "size": 5, "format": "json"}, max_pages=3)
# print(len(test), "résultats récupérés")
# test[0]


In [7]:
# 5. Récupération des stations piézométriques

params_stations = {
    "bbox": BBOX,           # zone géographique calculée à l'étape 3
    "size": PAGE_SIZE,      # nombre de résultats par page (configuration)
    "format": "json",
}
# NB : La documentation liste d'autres filtres possibles pour /stations, comme code_departement ou date_recherche.

stations = appeler_api(URL_STATIONS, params_stations)

print(f"\n{len(stations)} stations récupérées")


Page 1 : 176 résultats (176/176)

176 stations récupérées


In [8]:
# Visualiser le résultat :
stations[0]

{'code_bss': '07702X0081/1',
 'urn_bss': 'http://services.ades.eaufrance.fr/pointeau/07702X0081/1',
 'date_debut_mesure': None,
 'date_fin_mesure': None,
 'code_commune_insee': '26325',
 'nom_commune': "Saint-Rambert-d'Albon",
 'x': 4.83728378,
 'y': 45.289165654,
 'codes_bdlisa': None,
 'urns_bdlisa': [],
 'geometry': {'type': 'Point',
  'crs': {'type': 'name',
   'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
  'coordinates': [4.83728377989336, 45.2891656535815]},
 'bss_id': 'BSS001WLSH',
 'altitude_station': '164.0',
 'nb_mesures_piezo': 0,
 'code_departement': '26',
 'nom_departement': 'Drôme',
 'libelle_pe': None,
 'profondeur_investigation': 27.25,
 'codes_masse_eau_edl': None,
 'noms_masse_eau_edl': None,
 'urns_masse_eau_edl': [],
 'date_maj': 'Wed Jun 25 20:01:28 CEST 2025'}

### **DataFrame et contrôle contenu extraction :**
   6a. Création du DataFrame  
   6b. Contrôle qualité : types, valeurs manquantes, doublons  
   6c. Conversion des types  
   6d. Sélection des stations avec mesures

In [9]:
# 6a. Création du DataFrame

df_stations = pd.DataFrame(stations)

print("Dimensions (lignes, colonnes) :", df_stations.shape)
df_stations.head(3)

Dimensions (lignes, colonnes) : (176, 22)


,code_bss,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,x,y,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,07702X0081/1,http://services.ades.eaufrance.fr/pointeau/077...,NaN,NaN,26325,Saint-Rambert-d'Albon,4.837284,45.289166,None,[],...,164.0,0,26,Drôme,NaN,27.25,None,None,[],Wed Jun 25 20:01:28 CEST 2025
1,07477X0032/3821,http://services.ades.eaufrance.fr/pointeau/074...,NaN,NaN,38130,La Côte-Saint-André,5.274229,45.373098,None,[],...,349.0,0,38,Isère,NaN,20.50,None,None,[],Wed Jun 25 20:01:28 CEST 2025
2,08422X0190/F1,http://services.ades.eaufrance.fr/pointeau/084...,1999-12-17,2026-11-05,26166,Loriol-sur-Drôme,4.791298,44.756364,"[529AA00, 712BM01]",[http://reseau.eaufrance.fr/geotraitements/bdl...,...,95.0,9149,26,Drôme,PIEZOMETRE - AUTOROUTE (LORIOL SUR DROME - BRG...,11.50,[DG337],[Alluvions de la Drôme],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Sun Jul 14 13:00:02 CEST 2024


In [10]:
# 6b. Contrôle qualité : types, valeurs manquantes, doublons

# Valeurs manquantes par colonne (uniquement celles qui en ont)
manquants = df_stations.isna().sum()
print("\nValeurs manquantes :\n", manquants[manquants > 0].sort_values(ascending=False))
print("\n")
      
# Types des colonnes
df_stations.info()

# Doublons sur l'identifiant de station
print("\nDoublons sur code_bss :", df_stations["code_bss"].duplicated().sum())


Valeurs manquantes :
 date_debut_mesure           116
date_fin_mesure             116
noms_masse_eau_edl          111
codes_masse_eau_edl         111
codes_bdlisa                 91
libelle_pe                   41
profondeur_investigation     31
dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   code_bss                  176 non-null    str    
 1   urn_bss                   176 non-null    str    
 2   date_debut_mesure         60 non-null     str    
 3   date_fin_mesure           60 non-null     str    
 4   code_commune_insee        176 non-null    str    
 5   nom_commune               176 non-null    str    
 6   x                         176 non-null    float64
 7   y                         176 non-null    float64
 8   codes_bdlisa              85 non-null     object 
 9   urns_bdlisa               176 non-

In [11]:
# 6c. Conversion des types

df_stations["altitude_station"] = pd.to_numeric(df_stations["altitude_station"], errors="coerce")
df_stations["date_debut_mesure"] = pd.to_datetime(df_stations["date_debut_mesure"])
df_stations["date_fin_mesure"] = pd.to_datetime(df_stations["date_fin_mesure"])

df_stations[["altitude_station", "date_debut_mesure", "date_fin_mesure"]].dtypes


altitude_station            float64
date_debut_mesure    datetime64[us]
date_fin_mesure      datetime64[us]
dtype: object

In [12]:
# 6d. Sélection des stations ayant au moins une mesure de niveau d'eau

df_stations_mesurees = df_stations[df_stations["nb_mesures_piezo"] > 0].copy()

print(f"{len(df_stations_mesurees)} stations avec mesures sur {len(df_stations)}")
df_stations_mesurees[["code_bss", "nom_commune", "nom_departement",
                      "date_debut_mesure", "date_fin_mesure", "nb_mesures_piezo"]].head(10)


60 stations avec mesures sur 176


,code_bss,nom_commune,nom_departement,date_debut_mesure,date_fin_mesure,nb_mesures_piezo
2,08422X0190/F1,Loriol-sur-Drôme,Drôme,1999-12-17,2026-11-05,9149
4,07948X0047/F1,Romans-sur-Isère,Drôme,2009-06-12,2026-11-06,6129
5,08664X0069/F1,Soyans,Drôme,2008-12-03,2026-11-04,6519
6,07703X0064/P1,Anneyron,Drôme,2009-06-09,2026-04-20,6155
8,08187X0238/P1,Beaumont-lès-Valence,Drôme,2009-06-26,2026-11-06,6336
9,07955X0099/P1,Chatuzange-le-Goubet,Drôme,2005-11-24,2026-11-05,7442
12,08184X0084/PZ1,Valence,Drôme,1982-02-04,2026-10-21,9637
14,08422X0191/F2,Livron-sur-Drôme,Drôme,1999-12-17,2026-11-05,8462
17,08424X0006/F2,Eurre,Drôme,1995-06-29,2026-11-05,8892
20,08426X0133/P2,Saulce-sur-Rhône,Drôme,2009-06-17,2026-11-06,6242


In [13]:
# 7. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"stations_{date_extraction:%Y-%m-%d}.json"

# On enregistre les données ET le contexte de l'extraction (traçabilité)
contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_STATIONS,
    "params": params_stations,
    "nb_resultats": len(stations),
    "data": stations,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False, indent=2)

print(f"✅ {len(stations)} stations sauvegardées dans : {fichier_brut}")


✅ 176 stations sauvegardées dans : /home/virginie/Documents/DATA/00-PROJETS/Hub-eau/data/raw/stations_2026-11-10.json


In [14]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de stations :", len(relu["data"]))
assert len(relu["data"]) == len(stations), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

Date d'extraction : 2026-11-10T16:02:37
Nombre de stations : 176
